In [1]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

expert_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7
)

router_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

C:\Users\Meghana Veeramallu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """
You are a Senior Software Engineer.
Be precise, technical, and code-focused.
Explain bugs clearly and provide corrected code snippets.
"""
    },
    "billing": {
        "system_prompt": """
You are a Customer Billing Specialist.
Be empathetic and policy-driven.
Provide clear explanations about charges and refunds.
"""
    },
    "general": {
        "system_prompt": """
You are a friendly general AI assistant.
Handle casual and general questions politely.
"""
    },
    "tool": {
        "system_prompt": """
You are a financial assistant.
If crypto price data is provided, explain it clearly.
"""
    }
}

print("Experts Configured!")

Experts Configured!


In [4]:
from langchain_core.messages import SystemMessage, HumanMessage

def route_prompt(user_input: str) -> str:
    """
    Classifies into:
    technical | billing | general | tool
    """

    router_prompt = f"""
Classify this text into one of these categories:
[technical, billing, general, tool]

Rules:
- technical → programming, debugging
- billing → payments, refunds
- tool → real-time data like crypto price
- general → everything else

Return ONLY the category word.

Text:
"{user_input}"
"""

    response = router_llm.invoke([
        SystemMessage(content="You are a strict classifier."),
        HumanMessage(content=router_prompt)
    ])

    return response.content.strip().lower()

print("Router Ready.")

Router Ready.


In [5]:
def fetch_crypto_price():
    return "Bitcoin is currently trading at $62,450."

In [6]:
def process_request(user_input: str) -> str:

    category = route_prompt(user_input)
    print(f"[Router Selected]: {category}")
    
    if category == "tool":
        data = fetch_crypto_price()
        
        response = expert_llm.invoke([
            SystemMessage(content=MODEL_CONFIG["tool"]["system_prompt"]),
            HumanMessage(content=f"User asked: {user_input}\n\nData: {data}")
        ])
        
        return response.content
    
    if category not in MODEL_CONFIG:
        category = "general"
    
    response = expert_llm.invoke([
        SystemMessage(content=MODEL_CONFIG[category]["system_prompt"]),
        HumanMessage(content=user_input)
    ])
    
    return response.content

print("Orchestrator Ready.")

Orchestrator Ready.


In [7]:
queries = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription.",
    "What is the current price of Bitcoin?",
    "Tell me a joke."
]

for q in queries:
    print("\nUser:", q)
    result = process_request(q)
    print("Response:", result)


User: My python script is throwing an IndexError on line 5.
[Router Selected]: technical
Response: To assist you in debugging the issue, I'll need more information about your code. Please provide the following:

1. The exact error message you're seeing, including the line number and any relevant error details.
2. The relevant code snippet around line 5, including any loops or conditional statements that might be causing the issue.

That being said, an `IndexError` typically occurs when you're trying to access an element in a list or other indexed collection that doesn't exist. For example:

```python
my_list = [1, 2, 3]
print(my_list[3])  # Raises IndexError: list index out of range
```

In this case, the error occurs because the list only has three elements, but we're trying to access the fourth element (index 3).

To fix this issue, you can add a check to ensure that the index you're trying to access is within the valid range:

```python
my_list = [1, 2, 3]
if len(my_list) > 3:
    